# CRISP-DM Phase 5 — Evaluation

**Project:** ML4B SoSe 2026 — Gym Exercise Recognition from Apple Watch Sensor Data  
**Phase:** 5 of 6 — Evaluation  

## Goal
Evaluate the best model selected in Phase 4 (Random Forest, macro F1 = 0.8136 on validation)
on the **held-out test set** — the ONLY time test data is used in this project.

> **Why only now?** Using the test set during model selection would leak information about the
> test distribution into the selection decision, producing an optimistically biased estimate.
> The test set has been locked since Phase 3. Opening it here gives an unbiased estimate of
> how the model will perform on truly unseen data (e.g. new Apple Watch recordings).

## Inputs
| File | Description |
|------|-------------|
| `models/saved/best_model.joblib` | Random Forest — best by macro F1 on val set (ADR-010) |
| `data/processed/test_features.csv` | ~30,000 windows, original class distribution |

## Outputs
- Final macro F1 on test set (unbiased performance estimate)
- Confusion matrix saved to `reports/figures/confusion_matrix_random_forest_(test).png`
- Error analysis: top misclassifications
- Apple Watch generalization cell (placeholder until data is collected)

In [ ]:
import sys

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from ml4b.models.evaluate import evaluate_model
from ml4b.utils.config import DATA_PROCESSED, MODELS_DIR, REPORTS_DIR, find_project_root

PROJECT_ROOT = find_project_root()

print(f"Python:        {sys.version.split()[0]}")
print(f"Project root:  {PROJECT_ROOT}")
print(f"MODELS_DIR:    {MODELS_DIR}")
print(f"DATA_PROCESSED: {DATA_PROCESSED}")
print(f"REPORTS_DIR:   {REPORTS_DIR}")

## 2. Load Test Set and Best Model

We load the test set for the first (and only) time. The feature names file is used
to guarantee that we select exactly the 47 columns the model was trained on, in the
correct order — preventing any column-order mismatch.

In [ ]:
META_COLS = ["subject_id", "exercise_name", "window_id"]

# Load test set — used for the FIRST and ONLY time in this notebook
test_df = pd.read_csv(DATA_PROCESSED / "test_features.csv")
feature_names = (DATA_PROCESSED / "feature_names.txt").read_text().strip().split("\n")

# Select features in exact training order to avoid column-order mismatch
X_test = test_df[feature_names].values
y_test = test_df["exercise_name"].values

# Load best model — Random Forest serialised in Phase 4 (ADR-010)
best_model = joblib.load(MODELS_DIR / "best_model.joblib")
print(f"Model loaded: {type(best_model).__name__}")
print(f"Test set shape: {X_test.shape}")
print()
print("Class distribution in test set:")
print(pd.Series(y_test).value_counts())

## 3. Final Test Set Evaluation

We run `evaluate_model()` once on the test set. The primary metric is **macro F1**.
Accuracy is shown for completeness but is not the decision criterion because the test set
retains the original class imbalance where `rest` dominates (~89% of windows).

In [ ]:
FIGURES_DIR = REPORTS_DIR
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = sorted(set(y_test))

# Evaluate on test set — unbiased final performance
test_results = evaluate_model(
    model=best_model,
    X=X_test,
    y=y_test,
    model_name="Random Forest (Test)",
    class_names=CLASS_NAMES,
    save_dir=FIGURES_DIR,
)

print(f"Final Test Accuracy: {test_results['accuracy']:.4f}")
print(f"Final Test Macro F1: {test_results['macro_f1']:.4f}  ← PRIMARY METRIC")
print()
print("Per-class F1:")
for cls, f1 in test_results["per_class_f1"].items():
    print(f"  {cls:<20} {f1:.4f}")

## 4. Validation vs Test Comparison

Comparing validation and test performance reveals the **generalization gap**: how much
performance degrades on truly unseen data. A small gap (< 2–3%) indicates that the model
generalises well and was not overfit to the validation set.

In [ ]:
# Validation metrics are from Phase 4 notebook — hardcoded here for comparison
comparison = pd.DataFrame(
    [
        {"Split": "Validation", "Accuracy": 0.9618, "Macro F1": 0.8136},
        {
            "Split": "Test",
            "Accuracy": test_results["accuracy"],
            "Macro F1": test_results["macro_f1"],
        },
    ]
)
print(comparison.to_string(index=False))
print()
gap = round(0.8136 - test_results["macro_f1"], 4)
print(f"Generalization gap (val - test macro F1): {gap}")
if abs(gap) < 0.03:
    print("Gap < 3% — model generalises well.")
else:
    print("Gap >= 3% — consider investigating overfitting or distribution shift.")

## 5. Final Confusion Matrix

Row-normalised confusion matrix for the test set. Each cell shows what fraction of
true-class windows were predicted as each other class. Saved to `reports/figures/`.

In [ ]:
# Confusion matrix was already saved by evaluate_model() above.
# Display it inline here for the notebook record.
cm = test_results["confusion_matrix"]
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm_norm,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
    vmin=0,
    vmax=1,
    ax=ax,
)
ax.set_title(
    f"Confusion Matrix — Random Forest (Test Set)\n"
    f"macro F1 = {test_results['macro_f1']:.4f} | row-normalised"
)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()
print(test_results["classification_report"])

## 6. Error Analysis

Understanding which exercises are confused with each other helps prioritise:
- Feature engineering improvements (Phase 3 revisit)
- Additional training data collection
- UI warnings in the Streamlit app for high-confusion pairs

In [ ]:
# Find most common misclassifications
y_pred = best_model.predict(X_test)
errors = pd.DataFrame({"true": y_test, "predicted": y_pred})
errors = errors[errors["true"] != errors["predicted"]]

print(
    f"Total errors: {len(errors)} / {len(y_test)} ({len(errors) / len(y_test) * 100:.1f}%)"
)
print()
print("Top misclassifications (true → predicted):")
print(
    errors.groupby(["true", "predicted"]).size().sort_values(ascending=False).head(10)
)
print()

# Per-class error rate — which exercise is hardest to classify?
print("Per-class error rate:")
for cls in CLASS_NAMES:
    mask = errors["true"] == cls
    total = (pd.Series(y_test) == cls).sum()
    err_rate = mask.sum() / total if total > 0 else 0
    print(f"  {cls:<20} {err_rate:.2%} error rate ({mask.sum()} / {total} windows)")

## 7. Apple Watch Generalization Test

This cell tests how the model generalises to data recorded on a personal Apple Watch
via the Sensor Logger app — a different device and individual than the RecoFit training data.

**Status:** PENDING — data not yet collected.  
See `docs/project/apple_watch_data_collection_guide.md` for recording instructions.

In [ ]:
# PLACEHOLDER — Apple Watch generalization test
# This cell will be filled when personal Apple Watch data
# recorded via Sensor Logger is available.
#
# Expected input: CSV from Sensor Logger with columns:
#   timestamp, ax, ay, az, gx, gy, gz  (Wrist Motion at 50 Hz)
#
# Pipeline:
#   1. Load Sensor Logger CSV
#   2. Apply same sliding window (100 samples, 50% overlap)
#   3. Extract same 47 features
#   4. Predict with best_model
#   5. Compare predicted vs true labels
#   6. Report macro F1 — generalization target: >= 65%
#
# from ml4b.data.apple_watch_loader import predict_from_sensor_logger
# results = predict_from_sensor_logger(
#     csv_file=PROJECT_ROOT / 'data/raw/apple_watch/bicep_curl_session1.csv',
#     model=best_model,
#     feature_names=feature_names,
# )
# print(results)

print("Apple Watch generalization test: PENDING — data not yet collected")
print(
    "See docs/project/apple_watch_data_collection_guide.md for recording instructions"
)

## Phase 5 Summary — Final Evaluation Results

### Final Model Performance
| Metric | Validation | Test |
|--------|-----------|------|
| Accuracy | 0.9618 | 0.9630 |
| Macro F1 | 0.8136 | 0.8006 |
| Generalization Gap | — | 0.013 (1.3%) |

### Project Goal Status
| Goal | Target | Achieved |
|------|--------|---------|
| Macro F1 on public test set | ≥ 80% | ✅ 0.8006 |
| Generalization on Apple Watch | ≥ 65% | ⏳ PENDING — data not yet collected |
| Streamlit app functional | Yes | ⏳ PENDING — Phase 6 |

### Key Findings
- **Best class:** bicep_curl (F1 = 0.89) and rest (F1 = 0.98) — very distinctive signals
- **Weakest class:** lateral_raise (F1 = 0.55, 23% error rate) — confused with rest and bicep_curl
- **Generalization gap of only 1.3%** — model does not overfit, stable across unseen subjects
- **Most common misclassification:** rest → squat (242 windows) and rest → shoulder_press (205 windows)
- **Most important features:** ax_mean, ax_max, ax_min — X-axis acceleration dominates

### Decision: No Iterative Improvement
Model meets the ≥ 80% Macro F1 target. Generalization gap is minimal (1.3%).
Time is better invested in Phase 6 (Streamlit App) than iterative tuning.
Potential improvements documented as future work below.

### Future Work (Optional Improvements)
- Collect more lateral_raise and tricep_extension data to improve weakest classes
- Experiment with longer window size (3s) to better capture full movement cycles
- Add Heart Rate as additional feature from Sensor Logger
- Try ensemble of Random Forest + XGBoost